## 一、依赖库导入与环境配置

下面首先导入本实验所需的主要Python库，并进行环境变量与路径等基础配置。请确认已安装`tensorflow`、`sklearn`、`imblearn`等依赖。


In [1]:
import sys
import tensorflow as tf  # 深度学习框架
import numpy as np         # 数值计算
import pandas as pd        # 数据处理与分析
import matplotlib.pyplot as plt  # 可视化
import os                  # 文件与系统操作
import datetime            # 时间戳处理

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, LearningRateScheduler
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from tensorflow.keras.utils import to_categorical
from imblearn.over_sampling import SMOTE  # 数据平衡
from collections import Counter

## 二、超参数集中管理

采用面向对象方式封装所有可调超参数，方便后续实验对比与复现。

In [2]:
class Hyperparameters:
    """集中管理超参数"""
    def __init__(self):
        self.data_dir = "../preprocessed_data"      # 原始ECG数据文件夹
        self.labels_file = "labels.csv"             # 对应标签文件
        self.Kt = 8                                   # 卷积核大小
        self.pt = 0.5                                 # dropout概率
        self.Ft = 3                                   # 卷积核数量
        self.num_classes = 2                         # 类别数量（二分类）
        self.batch_size = 64                           # 训练批大小
        self.epochs = 100                             # 最大训练轮数
        self.learning_rate = 1e-3                     # 初始学习率
        self.patience_es = 16                         # EarlyStopping耐心轮数
        self.patience_lr = 8                          # 学习率调度耐心轮数
        self.factor_lr = 0.8                          # 学习率衰减因子
        self.min_lr = 1e-6                            # 最小学习率
        self.n_splits = 2                             # K折交叉验证折数
        self.random_state = 42                        # 随机种子


## 三、数据加载与预处理函数

定义加载单个ECG文件与批量预处理函数，包含12导联信号提取、归一化、标签编码与患者ID记录。

In [3]:
def load_ecg_data(file_path):
    """加载单文件ECG数据，返回12导联数组"""
    df = pd.read_csv(file_path)
    return df.iloc[:, 1:13].values  # 提取第2~13列导联信号


def preprocess_data(all_files, labels_df):
    """批量预处理所有ECG文件，输出样本、标签及患者ID"""
    ecg_samples = []
    labels = []
    patient_ids = []
    # 构建文件名到标签映射
    name_to_label = dict(zip(labels_df['name'], labels_df['labels']))
    for file_path in all_files:
        base_name = os.path.basename(file_path)
        name = base_name.split('_beat_')[0]
        if name not in name_to_label:
            continue
        try:
            df = pd.read_csv(file_path)
            ecg_data = df.iloc[:, 1:13].values
            max_val = np.max(np.abs(ecg_data))
            ecg_data = ecg_data / max_val  # 归一化至[-1,1]
            ecg_samples.append(ecg_data)
            labels.append(name_to_label[name])
            patient_ids.append(name)
        except FileNotFoundError:
            print(f"Warning: File not found - {file_path}")
            continue
    le = LabelEncoder()
    labels = le.fit_transform(labels)
    return np.array(ecg_samples), np.array(labels), np.array(patient_ids)

## 四、TCN模型构建

基于TensorFlow/Keras，按照论文结构依次堆叠时序卷积层、残差连接与激活，并输出分类结果。

In [4]:
def build_tcn_model(input_shape, hp):
    """构建具有多尺度扩张卷积的TCN模型"""
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv1D(8, hp.Kt, padding='causal', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    residual = layers.Conv1D(hp.Ft, 1, use_bias=False)(x)
    residual = layers.BatchNormalization()(residual)
    residual = layers.ReLU()(residual)
    for dilation_rate in [1, 2, 4]:
        y = layers.Conv1D(hp.Ft, hp.Kt, padding='causal', dilation_rate=dilation_rate, use_bias=False)(x)
        y = layers.BatchNormalization()(y)
        y = layers.ReLU()(y)
        y = layers.Dropout(hp.pt)(y)
        y = layers.Conv1D(hp.Ft, hp.Kt, padding='causal', dilation_rate=dilation_rate, use_bias=False)(y)
        y = layers.BatchNormalization()(y)
        y = layers.ReLU()(y)
        y = layers.Dropout(hp.pt)(y)
        x = layers.Add()([residual, y])
        x = layers.ReLU()(x)
        residual = x
    x = layers.Flatten()(x)
    activation = 'softmax' if hp.num_classes > 1 else 'sigmoid'
    outputs = layers.Dense(hp.num_classes, activation=activation)(x)
    return models.Model(inputs, outputs)


## 五、模型导出与可视化函数

包括导出模型架构摘要、训练及混淆矩阵绘制函数，便于实验结果记录与展示。

In [5]:
def export_model_architecture(model, filepath):
    with open(filepath, 'w') as f:
        model.summary(print_fn=lambda line: f.write(line + '\n'))
    print(f"模型架构已导出至: {filepath}")


def plot_training_subplots(histories, results_dir, n_splits):
    rows = int(np.ceil(n_splits/4)); cols = min(n_splits,4)
    fig, axes = plt.subplots(rows, cols, figsize=(16,4*rows), sharex='col', sharey='row')
    for i,h in enumerate(histories):
        ax = axes[i//cols,i%cols] if rows>1 else axes[i%cols]
        ax.plot(h.history['accuracy'], label='训练准确率')
        ax.plot(h.history['val_accuracy'], label='验证准确率')
        ax.set_title(f'第 {i+1} 折 准确率曲线')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.legend()
    plt.tight_layout(); plt.savefig(f'{results_dir}/kfold_training_accuracy.png'); plt.close()


def plot_learning_rate(histories, results_dir):
    """绘制学习率随训练轮次的变化曲线"""
    # 假设在回调中使用 LearningRateScheduler 并记录 'lr'
    fig = plt.figure(figsize=(6,4))
    for i,h in enumerate(histories):
        if 'lr' in h.history:
            epochs = range(1, len(h.history['lr'])+1)
            plt.plot(epochs, h.history['lr'], label=f'Fold {i+1}')
    plt.title('学习率变化曲线')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.legend()
    plt.tight_layout(); plt.savefig(f'{results_dir}/learning_rate.png'); plt.close()


def plot_confusion_matrices(cm_list, results_dir, n_splits, class_names=['0','1']):
    rows = int(np.ceil(n_splits/4)); cols = min(n_splits,4)
    fig, axes = plt.subplots(rows, cols, figsize=(12,3*rows))
    for i,cm in enumerate(cm_list):
        ax = axes[i//cols,i%cols] if rows>1 else axes[i%cols]
        im = ax.imshow(cm, cmap=plt.cm.Blues)
        ax.set_title(f'第 {i+1} 折 混淆矩阵')
        ax.set_xticks([0,1]); ax.set_yticks([0,1])
        ax.set_xticklabels(class_names); ax.set_yticklabels(class_names)
        for j in range(2):
            for k in range(2):
                ax.text(k,j,cm[j,k],ha='center',va='center',color='white' if cm[j,k]>cm.max()/2 else 'black')
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6)
    plt.tight_layout(); plt.savefig(f'{results_dir}/kfold_confusion_matrices.png'); plt.close()


## 六、训练评估核心函数

定义类别权重计算与模型评估流程，实现AUC计算、混淆矩阵与分类报告保存。


In [6]:
def calculate_class_weights(y_train):
    from sklearn.utils.class_weight import compute_class_weight
    classes = np.unique(y_train)
    weights = compute_class_weight('balanced', classes=classes, y=y_train)
    return dict(zip(classes, weights))


def evaluate_model(model, X_val, y_val, fold, summary_path, num_classes):
    print(f"\n评估第 {fold+1} 折模型...")
    y_prob = model.predict(X_val)
    if num_classes == 2:
        auc = roc_auc_score(y_val, y_prob[:,1]); y_pred = (y_prob[:,1]>0.5).astype(int)
        metrics = [f"AUC: {auc:.4f}"]
    else:
        y_pred = np.argmax(y_prob,axis=1); metrics = []
    cm = confusion_matrix(y_val, y_pred)
    report = classification_report(y_val, y_pred)
    with open(summary_path,'a',encoding='utf-8') as f:
        f.write(f"\n=== 第 {fold+1} 折评估 ===\n")
        f.write('\n'.join(metrics)+'\n')
        f.write(f"混淆矩阵:\n{cm}\n分类报告:\n{report}\n================\n")
    return cm


## 验证数据是否可分

In [7]:
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder

# def load_ecg_data(file_path):
#     """加载单文件ECG数据，返回12导联数组"""
#     df = pd.read_csv(file_path)
#     return df.iloc[:, 1:13].values  # 提取第2~13列导联信号

# def preprocess_data(all_files, labels_df):
#     """批量预处理所有ECG文件，输出样本、标签及患者ID"""
#     ecg_samples = []
#     labels = []
#     patient_ids = []
#     # 构建文件名到标签映射
#     name_to_label = dict(zip(labels_df['name'], labels_df['labels']))
#     for file_path in all_files:
#         base_name = os.path.basename(file_path)
#         name = base_name.split('_beat_')[0]
#         if name not in name_to_label:
#             continue
#         try:
#             df = pd.read_csv(file_path)
#             ecg_data = df.iloc[:, 1:13].values
#             max_val = np.max(np.abs(ecg_data))
#             ecg_data = ecg_data / max_val  # 归一化至[-1,1]
#             ecg_samples.append(ecg_data)
#             labels.append(name_to_label[name])
#             patient_ids.append(name)
#         except FileNotFoundError:
#             print(f"Warning: File not found - {file_path}")
#             continue
#     le = LabelEncoder()
#     labels = le.fit_transform(labels)
#     return np.array(ecg_samples), np.array(labels), np.array(patient_ids)

# def load_and_preprocess_data(all_files, labels_df):
#     """加载并预处理数据"""
#     X, y, patient_ids = preprocess_data(all_files, labels_df)
#     # 划分训练集和验证集
#     X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
#     return X_train, X_val, y_train, y_val

# hp = Hyperparameters()
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# all_files = [os.path.join(hp.data_dir, f) for f in os.listdir(hp.data_dir) if f.endswith('.csv')]
# labels_df = pd.read_csv(hp.labels_file, names=['name', 'labels'], header=0, encoding='utf-8')
# X, y, _ = preprocess_data(all_files, labels_df)
# X_train, X_val, y_train, y_val = load_and_preprocess_data(all_files, labels_df)
# print("训练集形状:", X_train.shape)  # (样本数, 时间步数, 12导联)
# print("类别分布:", np.bincount(y_train))



In [8]:

# from sklearn.preprocessing import FunctionTransformer


# def extract_statistical_features(X):
#     """提取每个导联的统计特征"""
#     features = []
#     for sample in X:
#         # 按导联计算均值、标准差、最大值
#         mean = np.mean(sample, axis=0)
#         std = np.std(sample, axis=0)
#         max_val = np.max(np.abs(sample), axis=0)
#         features.append(np.concatenate([mean, std, max_val]))
#     return np.array(features)

# # 提取特征
# X_train_flat = extract_statistical_features(X_train)
# X_val_flat = extract_statistical_features(X_val)
# print("特征提取后形状:", X_train_flat.shape)  # (样本数, 12*3=36)
# from sklearn.decomposition import PCA

# pca = PCA(n_components=2)
# X_train_pca = pca.fit_transform(X_train_flat)
# X_val_pca = pca.transform(X_val_flat)

# # 可视化
# import matplotlib.pyplot as plt

# plt.scatter(X_train_pca[y_train == 0, 0], X_train_pca[y_train == 0, 1], label="Class 0")
# plt.scatter(X_train_pca[y_train == 1, 0], X_train_pca[y_train == 1, 1], label="Class 1")
# plt.legend()
# plt.title("PCA Visualization of ECG Classes")
# plt.show()

### 判断线性可分

In [9]:
# from sklearn.linear_model import Perceptron
# from sklearn.svm import LinearSVC
# from sklearn.metrics import accuracy_score

# # 感知机
# perceptron = Perceptron(max_iter=1000, tol=1e-3)
# perceptron.fit(X_train_flat, y_train)
# y_pred = perceptron.predict(X_val_flat)
# print("Perceptron Accuracy:", accuracy_score(y_val, y_pred))

# # 线性SVM
# linear_svm = LinearSVC(max_iter=10000)
# linear_svm.fit(X_train_flat, y_train)
# print("Linear SVM Accuracy:", linear_svm.score(X_val_flat, y_val))

### 是否非线性可分

In [10]:
# from sklearn.svm import SVC

# rbf_svm = SVC(kernel='rbf', gamma='scale')
# rbf_svm.fit(X_train_flat, y_train)
# print("RBF SVM Accuracy:", rbf_svm.score(X_val_flat, y_val))

### 特征重要性

In [11]:
# from sklearn.inspection import permutation_importance

# # 使用随机森林计算特征重要性
# from sklearn.ensemble import RandomForestClassifier
# print("X_train_flat:", X_train_flat.shape)
# print("y_train:", y_train.shape)
# print("X_train:", X_train.shape)
# rf = RandomForestClassifier()
# rf.fit(X_train_flat, y_train)
# importances = permutation_importance(rf, X_val_flat, y_val, n_repeats=10)
# print("Feature Importances:", importances.importances_mean)

## 七、主流程入口

整合数据载入、交叉验证训练、评估与结果可视化，最终生成实验报告与图像。

In [12]:
def main():
    hp = Hyperparameters()
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    res_dir = os.path.join("results",ts); os.makedirs(res_dir,exist_ok=True)
    summary = os.path.join(res_dir,'summary.txt')

    # 加载与预处理
    files = [os.path.join(hp.data_dir,f) for f in os.listdir(hp.data_dir) if f.endswith('.csv')]
    df_lbl = pd.read_csv(hp.labels_file,names=['name','labels'],header=0,encoding='utf-8')
    X,y,_ = preprocess_data(files,df_lbl)
    with open(summary,'w',encoding='utf-8') as f:
        f.write(f"时间: {ts}\n分布: {dict(zip(*np.unique(y,return_counts=True)))}\n")

    # 可分性探索
    # plot_separability(X, y, res_dir)

    # 构建与导出架构
    model0 = build_tcn_model(X.shape[1:],hp)
    export_model_architecture(model0, os.path.join(res_dir,'model.txt'))

    # 交叉验证训练与评估
    skf = StratifiedKFold(n_splits=hp.n_splits, shuffle=True, random_state=hp.random_state)
    histories, cms = [], []
    for i,(tr,va) in enumerate(skf.split(X,y)):
        print(f"\nFold {i+1}")
        Xtr, Xva = X[tr], X[va]; ytr, yva = y[tr], y[va]
        print(f"原始: {Counter(ytr)}")
        sm = SMOTE(random_state=hp.random_state)
        Xf = Xtr.reshape(len(tr),-1); Xr, yr = sm.fit_resample(Xf,ytr)
        Xr = Xr.reshape(-1,*Xtr.shape[1:]); print(f"平衡: {Counter(yr)}")
        m = build_tcn_model(X.shape[1:],hp)
        m.compile(tf.keras.optimizers.Adam(hp.learning_rate),'sparse_categorical_crossentropy',['accuracy'])
        # 增加 LearningRateScheduler 回调以记录 lr 历史
        lr_cb = LearningRateScheduler(lambda epoch, lr: lr, verbose=0)
        cb = [EarlyStopping('val_accuracy',hp.patience_es,restore_best_weights=True),
              ReduceLROnPlateau('val_loss',hp.factor_lr,hp.patience_lr,hp.min_lr),
              ModelCheckpoint(os.path.join(res_dir,f"m{i+1}.h5"),'val_accuracy',save_best_only=True),
              lr_cb]
        h = m.fit(Xr,yr,epochs=hp.epochs,batch_size=hp.batch_size,validation_data=(Xva,yva),callbacks=cb,verbose=1)
        histories.append(h)
        cms.append(evaluate_model(m,Xva,yva,i,summary,hp.num_classes))

    # 可视化结果
    plot_training_subplots(histories, res_dir, hp.n_splits)
    plot_learning_rate(histories, res_dir)
    plot_confusion_matrices(cms, res_dir, hp.n_splits)

In [13]:
if __name__ == "__main__":
    main()

模型架构已导出至: results\20250615_105201\model.txt

Fold 1
原始: Counter({1: 6757, 0: 2331})
平衡: Counter({1: 6757, 0: 6757})
Epoch 1/100
212/212 [==============================] - 13s 17ms/step - loss: 1.0101 - accuracy: 0.5972 - val_loss: 1.1164 - val_accuracy: 0.2661

评估第 1 折模型...


d:\Anaconda3\envs\tf_0530\lib\site-packages\keras\utils\generic_utils.py:494: CustomMaskWarning: Custom mask layers require a config and must override get_config. When loading, the custom mask layer must be passed to the custom_objects argument.
  warnings.warn('Custom mask layers require a config and must override '



Fold 2
原始: Counter({1: 6757, 0: 2331})
平衡: Counter({1: 6757, 0: 6757})
Epoch 1/100
212/212 [==============================] - 6s 17ms/step - loss: 1.1186 - accuracy: 0.5568 - val_loss: 1.0632 - val_accuracy: 0.2837

评估第 2 折模型...


d:\Anaconda3\envs\tf_0530\lib\site-packages\keras\utils\generic_utils.py:494: CustomMaskWarning: Custom mask layers require a config and must override get_config. When loading, the custom mask layer must be passed to the custom_objects argument.
  warnings.warn('Custom mask layers require a config and must override '
d:\Anaconda3\envs\tf_0530\lib\site-packages\matplotlib\backends\backend_agg.py:240: RuntimeWarning: Glyph 31532 missing from current font.
  font.set_text(s, 0.0, flags=flags)
d:\Anaconda3\envs\tf_0530\lib\site-packages\matplotlib\backends\backend_agg.py:240: RuntimeWarning: Glyph 25240 missing from current font.
  font.set_text(s, 0.0, flags=flags)
d:\Anaconda3\envs\tf_0530\lib\site-packages\matplotlib\backends\backend_agg.py:240: RuntimeWarning: Glyph 20934 missing from current font.
  font.set_text(s, 0.0, flags=flags)
d:\Anaconda3\envs\tf_0530\lib\site-packages\matplotlib\backends\backend_agg.py:240: RuntimeWarning: Glyph 30830 missing from current font.
  font.set_tex